### Tutorial 6 — LoRA fine-tuning KRONOS2 for cell phenotyping

Tutorial 5 measured what a **frozen** KRONOS2 embedding knows about cell type: freeze the encoder, fit a logistic regression, report the score. That number is a property of the *representation*.

This tutorial asks the next question: *how much better does it get if the encoder is allowed to be finetuned for this task?*

Fully fine-tuning a ViT-Base means training 86M parameters on a few thousand cells, which risks overfitting and losing the pretrained representation. **LoRA** (Low-Rank Adaptation) is the standard answer. Every adapted `nn.Linear` keeps its frozen weight and learns a low-rank update. Adapting the last 6 of 12 blocks at rank 8 costs **602,128 trainable parameters — 0.7% of the backbone**.

**How this tutorial is organised.** The training could take some time per fold, so it runs in a small script, [`finetune_cells.py`](finetune_cells.py), that you launch from a terminal (tmux / nohup) and leave running. This notebook does everything *around* that:

1. **Reuse Tutorial 5's folds** — the same spatial quadrants, the same cells, read from the CSVs it wrote
2. **Set the configuration** — rank, scaling, how many blocks to adapt, learning rate, and the data budget
3. **Inject the adapters** — 602k parameters into the last 6 blocks
4. **Launch training** from the command line (step 5), then come back
5. **Evaluate** the held-out quadrants from what the script wrote
6. **Compare** against the frozen linear probe *on identical rows* — same folds, same cells, same budget

The heavy machinery — a fold-aware `Dataset`, the class-balanced sampler, the training loop, the metrics — lives in [`utils/lora_finetune.py`](utils/lora_finetune.py), shared by this notebook and the script, so the cells below stay about the *protocol*.

> **This is an advanced tutorial.** It assumes you have run Tutorials 0–5. Unlike Tutorials 1–5 it **requires a GPU** — around 13 GB of VRAM at the default batch size.

> **Prerequisites.** [Tutorial 0](0-Example-Data-Download.ipynb) (data), [Tutorial 1](1-Tissue-Ingest.ipynb) (ingest), [Tutorial 3](3-Cell-Segmentation-and-Feature-Extraction.ipynb) (cell mask, labels, KRONOS2 cell features), and [Tutorial 5](5-Cell-Phenotyping.ipynb) — specifically its step that writes `example-data/cell-pheno-results/folds/` and `best_c.csv`. This notebook reads those and will tell you if they are missing.

> This applies to cell phenotyping applications only. For slide-level prognostication tasks, the number of training data points are very small for finetuning KRONOS2

#### 0 — Requirements

Fine-tuning needs the `kronos2` extra (torch + transformers) and scikit-learn for the baseline probe:

```bash
uv sync --extra kronos2
uv pip install scikit-learn
```

> **Run everything through `uv run`.** `uv sync` installs into the project's `.venv/`; it does not touch the interpreter your shell's bare `python` resolves to (a conda base environment, typically). So `python tutorials/finetune_cells.py` will fail with `ModuleNotFoundError: No module named 'torch'` even after a successful sync. Prefix commands with `uv run` — as every shell block below does — or activate the environment once per shell with `source .venv/bin/activate` and drop the prefix. Same for Jupyter: launch it as `uv run jupyter lab`, or select the `.venv` kernel.

**Where the time goes.** Only step 5 (the training script) is expensive — roughly **3 hours per fold** on a single RTX 3090 at Tutorial 5's full budget, so on the order of **half a day for all four folds**. Everything else is cheap. So you can read straight through, launch training when you reach step 5, and return to steps 6–7 once it finishes.

#### 1. Reuse Tutorial 5's folds

We want the finetuning results to be **comparable** to Tutorial 5. That means not re-deriving the split: Tutorial 5 wrote its spatial quadrant folds to disk precisely so downstream experiments could load the same rows instead of re-deriving quadrants and hoping the seed matched.

So we read those CSVs. Guard band, quadrant assignment, validation split — all of it is inherited. `FoldPatches.from_slide` does the wiring: it reads the ordered marker list and patch slug off Tutorial 3's stored feature array, opens the patch-set coords and cell ids, and collects the global class order from every fold CSV.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make the tutorial's utils/ importable whether Jupyter launched from
# tutorials/ or from the repo root.
cwd = Path.cwd()
tutorials_dir = cwd if (cwd / "utils").exists() else cwd / "tutorials"
sys.path.insert(0, str(tutorials_dir))

from utils.lora_finetune import FinetuneConfig, FoldPatches

from coral import CoralSlide
from coral.config import PatchConfig
from coral.config.subset import Selection

OUTPUT_DIR = tutorials_dir / "example-data" / "processed"
FOLDS_DIR = tutorials_dir / "example-data" / "cell-pheno-results" / "folds"
RESULTS_DIR = tutorials_dir / "example-data" / "cell-pheno-results" / "lora"

CELL_PATCH_SIZE = 64   # must match Tutorial 3

# The 18 phenotypic markers Tutorial 3 extracted on.
PANEL = [
    "dapi", "cd11b", "cd11c", "cd15", "cd163", "cd20", "cd206", "cd30",
    "cd31", "cd4", "cd56", "cd68", "cd7", "cd8", "cytokeratin", "foxp3",
    "mct", "podoplanin",
]

if not FOLDS_DIR.exists():
    raise RuntimeError(
        f"No folds at {FOLDS_DIR}.\nRun Tutorial 5 (Cell Phenotyping) "
        "first — its step 3 writes the train/val/test CSVs this "
        "notebook reuses."
    )

slide = CoralSlide.open(OUTPUT_DIR / "raw_image.zarr")
cell_cfg = PatchConfig(patch_size=CELL_PATCH_SIZE, mode="cell_centered")
panel = Selection(include=PANEL)
fold_patches = FoldPatches.from_slide(slide, cell_cfg, panel, FOLDS_DIR)

print(slide)
print(f"{len(fold_patches.class_names)} classes: {fold_patches.class_names}")
print(f"markers ({len(fold_patches.markers)}): {fold_patches.markers}")
print(f"nuclear : {fold_patches.nuclear!r}  (the model's DAPI special-case)")
for fold in (1, 2, 3, 4):
    n_tr = len(fold_patches.load_split(f"train_2000_fold{fold}.csv")[0])
    n_va = len(fold_patches.load_split(f"val_fold{fold}.csv")[0])
    n_te = len(fold_patches.load_split(f"test_fold{fold}.csv")[0])
    print(f"fold {fold}: train {n_tr:>6,}  valid {n_va:>6,}  test {n_te:>6,}")

#### 2. Why the backbone is called directly

KRONOS2's public forward is decorated `@torch.inference_mode()`. That is exactly right for an inference-only feature extractor — and it makes gradients impossible:

```python
cls = model(patches, markers)   # no autograd graph, cannot backpropagate
```

Training therefore calls the backbone's `forward_features` instead, which is the same computation without the inference guard:

```python
model.backbone.forward_features(x, masks=None, marker_names=...)["x_norm_clstoken"]
```

Take a look inside `utils/lora_finetune.py` for functionalities related for LoRA finetuning.

#### 3. The configuration

Every knob for a run lives in one place: `FinetuneConfig` in [`utils/lora_finetune.py`](utils/lora_finetune.py).

**The adapter shape — `lora_rank` and `lora_alpha`.** These two define LoRA. The rank `r = 8` is the inner dimension of the update `Δ = (α/r)·BA`: it sets how much the adapter can express, and the parameter cost is linear in it. `lora_alpha = 16` is the scaling *numerator*, so the effective multiplier on the update is `α/r = 2`. Splitting one scale into two numbers looks redundant until you sweep `r` — the convention `α = 2r` holds the update's magnitude fixed while capacity changes, so a rank sweep doesn't silently become a learning-rate sweep at the same time. `lora_dropout = 0.0`: at 602k parameters on a few thousand cells, weight decay and early stopping already cover that ground.

**Where they go — `num_lora_blocks = 6`.** Adapt the deepest 6 of KRONOS2's 12 transformer blocks, those that are closest to the representations. The deep blocks carry the task-specific representation and are the productive place to spend capacity; early blocks encode generic structure worth leaving frozen. This is the second capacity lever, and it trades against `r` — step 4 works out what the two cost together, and where they land in the backbone.

**The optimizer — `lr = 6e-4`, `weight_decay = 0.01`.** That learning rate is well above what a full fine-tune of this backbone would tolerate, and deliberately so: only 0.7% of the parameters move, and `B` is zero-initialised, so training starts exactly at the pretrained function rather than at a perturbed one. There is no pretrained behaviour to wreck in the first few steps, so the adapters can afford to move fast. AdamW updates the adapters and the classification head together as one parameter group, on a cosine schedule over `epochs = 20`, with `patience = 5` early stopping on validation loss.

**Runtime knobs.** `batch_size = 128` is ~13 GB of VRAM. `num_workers = 8` matters more than it looks: reading a cell patch costs ~24 ms, roughly 3.5× the GPU time of the step that consumes it, so loading — not compute — is the bottleneck. A `DataLoader`'s workers are separate processes, so the dataset has to survive being sent to each one. `CoralDataset` stores only a path and opens zarr inside every read, so each worker opens its own handle rather than sharing a live one — which is what lets the reads run in parallel, worth ~2.5× end to end. Past 8 workers the reads are no longer the bottleneck and the gain flattens.

**The data budget matches Tutorial 5 exactly.** `max_cells_per_class = None` and `max_valid_per_class = None` mean the fold CSVs are used as Tutorial 5 wrote them: 2000 training cells per class, and the full validation split. That is what lets step 7 score both arms on identical rows; cap either flag and the two are no longer comparable.

In [ ]:
config = FinetuneConfig()   # the tutorial defaults; the script's defaults too
config

#### 4. Inject the LoRA adapters

`build_classifier` turns those settings into a model. It freezes the backbone first, then wraps the target layers — order matters, since freezing after injection would freeze the adapters too. It returns the adapted backbone, a fresh linear head on the CLS token, the names of the wrapped layers, and the trainable-parameter count.

The adapter geometry is worth a moment. Each block contributes `r · (in + out)` parameters per adapted linear layer. On ViT-Base (`d = 768`) the four targets cost `12,288 · r` per block:

| target | in → out | cost at r=8 |
| --- | --- | --- |
| `attn.qkv` | 768 → 2304 | 24,576 |
| `attn.proj` | 768 → 768 | 12,288 |
| `mlp.fc1` | 768 → 3072 | 30,720 |
| `mlp.fc2` | 3072 → 768 | 30,720 |

That is 98,304 per block, so 6 blocks = 589,824, plus a `768 × 16` head — the 602k the introduction quoted. Note that KRONOS2 stores its blocks in 4 chunks (inheriting from DINOv2 implementation, with each chunk containing 3 blocks), so the helper locates blocks **structurally** — any module exposing both `attn` and `mlp`.

The cell below builds a model only to read the shape of the thing off it, then releases it — training (step 5) builds its own.

In [ ]:
import torch

from utils.lora_finetune import build_classifier

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: no GPU found — fine-tuning on CPU is impractically slow.")

backbone, head, wrapped, n_trainable = build_classifier(
    DEVICE, len(fold_patches.class_names), config
)
n_total = sum(p.numel() for p in backbone.parameters())
print(f"wrapped {len(wrapped)} layers across {config.num_lora_blocks} blocks")
print(f"  first: {wrapped[0]}\n  last : {wrapped[-1]}")
print(f"trainable: {n_trainable:,} of {n_total:,}  "
      f"({100 * n_trainable / n_total:.2f}% of the backbone)")

del backbone, head
torch.cuda.empty_cache()

#### 5. Train — outside the notebook

This is the one expensive step, and it does **not** run in the notebook. Instead, launch [`finetune_cells.py`](finetune_cells.py) from a terminal in a detachable session and leave it running.

Run these from the **repo root**, so `uv run` finds the project (see step 0 — a bare `python` here is the usual cause of `No module named 'torch'`):

```bash
# all four folds, inside a tmux (or screen) session so it survives a disconnect:
tmux new -s lora
uv run python tutorials/finetune_cells.py --folds 1,2,3,4 --num-workers 8
#   detach with Ctrl-b d; reattach later with `tmux attach -t lora`
```

The folds run sequentially and each writes its outputs as it finishes, so a crash on fold 3 does not cost you folds 1 and 2.

Every `FinetuneConfig` field has a matching flag — run `uv run python tutorials/finetune_cells.py --help` to see them. For a quick end-to-end test, shrink everything to one short fold:

```bash
uv run python tutorials/finetune_cells.py --folds 1 --epochs 1 \
    --max-cells-per-class 200 --num-workers 4
```

For each fold, the script writes into `example-data/cell-pheno-results/lora/`:

| file | contents |
| --- | --- |
| `lora_fold{f}.pt` | the adapter + head checkpoint (~2.4 MB — only the trainable state) |
| `preds_fold{f}.npz` | held-out truths + probabilities |
| `history_fold{f}.csv` | per-epoch train / val loss and balanced accuracy |
| `lora_results.csv` | combined per-fold metrics |

The rest of this notebook reads those back. **Run the training now**, then continue once all four folds have finished.

#### 6. Evaluate across the four folds

The training script wrote one `preds_fold{f}.npz` per fold — held-out truths and probabilities — so evaluation is just reading those back. The same four metrics Tutorial 5 reports, computed the same way: `score` scatters probabilities onto the fixed global class order so every column means the same class in every fold.

Report the result as **mean ± standard deviation** across the four held-out quadrants, not as a single number. Each fold holds out a different region of tissue, so the spread is what tells you whether a result survives a change of region — and it is the yardstick the comparison in step 7 has to clear. A single fold gives you a point estimate with no way to judge it.

The script also wrote `history_fold{f}.csv` with per-epoch train/validation loss and balanced accuracy. This notebook does not plot it, but it is there if you want to inspect convergence — read it with the caveat that training loss is computed on class-balanced batches while validation is the real, imbalanced distribution, so the two are not on the same scale and only their *trends* are comparable.

In [ ]:
from utils.lora_finetune import score


def load_preds(fold):
    data = np.load(RESULTS_DIR / f"preds_fold{fold}.npz")
    return data["truth"], data["probs"]


def summarize(results, name):
    """Per-fold metrics, plus mean and standard deviation across folds."""
    summary = pd.DataFrame(
        [results.mean(), results.std(ddof=0)], index=["Mean", "Std Dev"]
    )
    out = pd.concat([results, summary]).round(4)
    out.index.name = name
    return out


FOLDS = tuple(
    int(p.stem.split("fold")[-1])
    for p in sorted(RESULTS_DIR.glob("preds_fold*.npz"))
)
if not FOLDS:
    raise RuntimeError(
        f"No predictions in {RESULTS_DIR}.\nRun the training script "
        "(step 5) first — it writes preds_fold*.npz."
    )
if len(FOLDS) < 4:
    print(
        f"WARNING: only folds {list(FOLDS)} on disk — the spread below is "
        "not meaningful. Train all four with --folds 1,2,3,4."
    )

LABELS = fold_patches.labels

lora_rows = []
for fold in FOLDS:
    truth, probs = load_preds(fold)
    metrics = score(truth, probs, LABELS)
    lora_rows.append({"Fold": f"fold_{fold}", **metrics})

lora_results = pd.DataFrame(lora_rows).set_index("Fold")
summarize(lora_results, "LoRA fine-tune")

#### 7. The controlled comparison

Now the question the notebook exists to answer: **is adapting the encoder worth it, against simply probing it frozen?** For fair comparison, we load linear probing results we obtained from Tutorial 5 with the best `C` per fold. The train, validation, and test datasets are exactly the same across both evaluation approaches.

In [ ]:
from utils.lora_finetune import frozen_probe

BEST_C_PATH = FOLDS_DIR.parent / "best_c.csv"
if not BEST_C_PATH.exists():
    raise RuntimeError(
        f"No best_c.csv at {BEST_C_PATH}.\nRun Tutorial 5 (Cell "
        "Phenotyping) first — its step 8 writes the per-fold C this "
        "notebook inherits."
    )

# Per-fold C from Tutorial 5's Optuna search — inherited, not re-searched.
BEST_C = pd.read_csv(BEST_C_PATH, index_col="Fold")["KRONOS2"]

frozen_rows, frozen_preds = [], {}
for fold in FOLDS:
    best_c = float(BEST_C.loc[f"fold_{fold}"])
    truth, probs = frozen_probe(fold_patches, best_c, fold)
    frozen_preds[fold] = (truth, probs)
    metrics = score(truth, probs, LABELS)
    frozen_rows.append({"Fold": f"fold_{fold}", **metrics})
    print(f"fold {fold} (C={best_c:.1e}, from Tutorial 5): " +
          ", ".join(f"{k}={v:.4f}" for k, v in metrics.items()))

frozen_results = pd.DataFrame(frozen_rows).set_index("Fold")
summarize(frozen_results, "frozen probe")

In [ ]:
comparison = pd.DataFrame({
    "frozen probe": frozen_results.mean(),
    "LoRA fine-tune": lora_results.mean(),
})
comparison["Δ"] = comparison["LoRA fine-tune"] - comparison["frozen probe"]
comparison.round(4)